# Walkthrough: lightweight fake news detectionA guided tour of the `fakenews` package — the corpus, the label convention, thetwo base models and the ensemble.This notebook is for *understanding* the pipeline. To reproduce the paper'snumbers, run the scripts in `experiments/` instead; they are seeded, they recordtheir configuration, and they write machine-readable results.**Prerequisites**```bashpip install -e ".[app,bench,dev]"fakenews build-cache        # cleans and POS-tags the corpus once (~20 min)```

In [ ]:
import logginglogging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s | %(message)s")logging.getLogger("tensorflow").setLevel(logging.ERROR)from fakenews.config import CLASS_NAMES, ExperimentConfigfrom fakenews.data.cache import load_cached_corpusconfig = ExperimentConfig()config

## 1. The corpus and its label conventionThe dataset description says `Label 0 = fake and 1 = real`, and in the samebreath reports 35,028 real and 37,106 fake articles. The counts below settle thecontradiction: **0 is legitimate, 1 is fake**.See `docs/label-semantics.md` for why this changes per-class reporting only, andcannot invalidate anyone's published accuracy.

In [ ]:
texts, labels, pos_features = load_cached_corpus(blocklist_preset="none")print(f"{len(texts):,} articles")for value, name in enumerate(CLASS_NAMES):    print(f"  label {value} = {name:6s}  {(labels == value).sum():,}")

## 2. Do the POS features separate the classes?The paper hypothesises that fake news carries more adjectives and adverbs andfewer nouns. With ~72,000 documents almost any difference is statisticallysignificant, so the effect size is what matters, not the p-value.

In [ ]:
from fakenews.evaluation.significance import class_conditional_effectfrom fakenews.features.pos import POS_FEATURE_COLUMNSfor index, column in enumerate(POS_FEATURE_COLUMNS):    effect = class_conditional_effect(pos_features[column].to_numpy(), labels, feature=column)    print(        f"{column.replace('pos_', ''):11s} "        f"legit {effect.mean_legit:.4f}   fake {effect.mean_fake:.4f}   "        f"d = {effect.cohens_d:+.3f}"    )

## 3. Source leakageWELFake's real-news portion inherits `(Reuters)` datelines from its ISOT source.Cleaning strips the brackets and leaves a bare, highly discriminative token. Thecache materialises blocklisted variants so the effect can be measured rather thanargued about.

In [ ]:
stripped, _, _ = load_cached_corpus(blocklist_preset="wire")position = next(i for i in range(len(texts)) if " reuters " in f" {texts.iloc[i]} ")print("unmodified :", texts.iloc[position][:120])print("wire-free  :", stripped.iloc[position][:120])

## 4. Train and evaluateOne call runs the whole pipeline: stratified split, TF-IDF + SVD, POSconcatenation, XGBoost, GloVe-backed TextCNN, and all three ensembles.On a CPU this takes roughly 20–30 minutes.

In [ ]:
from fakenews.pipeline import run_pipelineresult = run_pipeline(config, save_results=False, verbose=0)print(result.summary_table())

## 5. Is the ensemble's margin real?The ensemble's lead over TextCNN is a fraction of a percentage point. A pairedMcNemar test asks the right question: do the two models disagree *asymmetrically*,or do they simply make a similar number of errors in different places?

In [ ]:
from fakenews.evaluation.significance import mcnemar_test, paired_difference_citest = mcnemar_test(    result.y_test,    result.predictions["ensemble_soft_voting"],    result.predictions["textcnn"],    model_a="Ensemble",    model_b="TextCNN",)difference, low, high = paired_difference_ci(    result.y_test,    result.predictions["ensemble_soft_voting"],    result.predictions["textcnn"],)print(f"ensemble right / cnn wrong : {test.n_a_only}")print(f"cnn right / ensemble wrong : {test.n_b_only}")print(f"accuracy difference        : {difference:+.4f}  95% CI [{low:+.4f}, {high:+.4f}]")print(f"McNemar p                  : {test.p_value:.4g}")

## 6. Where the errors fallBecause the positive class is ambiguous in this dataset, the false positive rateis computed from the confusion matrix rather than as `1 - recall` — recall ismeasured on the *fake* class, so the two are different quantities.

In [ ]:
metrics = result.results["ensemble_soft_voting"]print(f"false positives (legit flagged fake) : {metrics.false_positives}")print(f"false negatives (fake missed)        : {metrics.false_negatives}")print(f"false positive rate                  : {metrics.false_positive_rate:.4f}")print(f"recall on the fake class             : {metrics.recall:.4f}")

## 7. Classifying a new article`FakeNewsDetector` owns the whole path from raw title and text to a labelledprobability, so the served model cannot drift from the trained one. It is whatthe Flask app in `app/` uses.```pythonfrom fakenews.detector import FakeNewsDetectordetector = FakeNewsDetector.load("artifacts/models/seed5")detector.predict_article(title="...", text="...")```Train and save a model first with `python experiments/run_main.py`.